  # Detecting profiles in a population using K-modes clustering on qualitative variables

### Add kmodes library to pip environment if missing

* Open a terminal and activate your data analysis environment:

    -> my_venvs_activate data_analysis
* Install the missing package in the active environment:

    -> pip install kmodes

* Deactivate the environment when you have finished:

    -> my_venvs_deactivate


In [ ]:
### Activate libraries that will be used in the notebook

import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
import fanalysis.ca as fa 
from kmodes.kmodes import KModes
from sklearn.metrics import silhouette_score

import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

import networkx as nx

import sqlite3 as sql



In [ ]:
### Libraries used to import local modules

import sys
from importlib import reload


# Add parent directory to the path
sys.path.insert(0, '..')

### If you want to add the parent-parent directory,
sys.path.insert(0, '../..')



In [ ]:

import bivariate_library as bl
import correspondence_analysis_library as cal
import cluster_functions as cf


In [ ]:
### Use this to reload the functions if modified
#print(reload(bl))
print(reload(cf))  

In [ ]:
import warnings
warnings.filterwarnings('ignore')


## Create a dataframe with the data to be analysed

In this notebook, we use the data produced with the [bivariate analysis notebook using countries](da3-1_countries_bivariate_analysis.ipynb) and the data collected in the [da5-employer.md](../../documentation/wikidata/data-analysis/da5-employer.md) and [da5-employer.sql](../../documentation/wikidata/data-analysis/da5-employer.sql) files.


The data was prepared and exported in the [da5_MCA]() notebook: "Prepare file for cluster analysis".




In [ ]:
file_address='da_data/da5-MCA-clusters.csv'
df_pm = pd.read_csv(file_address)
df_pm.head(3)

In [ ]:
### Inspect the dataframe and 
# notably if there are missing values
df_pm.info()

In [ ]:
### Use this in Noto to keep seeing all dataframe columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
# Reset to default settings if needed later
pd.reset_option('display.max_columns')

## Inspect the distribution of the variables

As mentioned above, the data was prepared in a former notebook. 

The variables have already been coded and limited to a reasonable number of categories. The number of categories depends, of course, on the research questions.

In [ ]:
### Group and count: gender
# This variable creates a significant divide
df_count = df_pm.groupby('gender').size()
df_count = pd.DataFrame(df_count.sort_values(ascending = False))
df_count.columns=['number']
print(df_count.iloc[:50])



In [ ]:
### Group and count: main occupation
# This variable creates a significant divide
df_count = df_pm.groupby('occupation_main').size()
df_count = pd.DataFrame(df_count.sort_values(ascending = False))
df_count.columns=['number']
print(df_count.iloc[:50])



In [ ]:
### Group and count: country
df_count = df_pm.groupby('coded_country').size()
df_count = pd.DataFrame(df_count.sort_values(ascending = False))
df_count.columns=['number']
print(df_count.iloc[:70])

In [ ]:
### Group and count: secondary occupation
# We observe some dispersion that requires grouping the categories of the variable
df_count = df_pm.groupby('occupation_sec1').size()
df_count = pd.DataFrame(df_count.sort_values(ascending = False))
df_count.columns=['number']
print(df_count.iloc[:70])

In [ ]:
### Group and count: coded employer
# We observe some dispersion that requires grouping the categories of the variable
df_count = df_pm.groupby('coded_employer').size()
df_count = pd.DataFrame(df_count.sort_values(ascending = False))
df_count.columns=['number']
print(df_count.iloc[:70])

In [ ]:
### Group and count: activity periods
# We observe some dispersion that requires grouping the categories of the variable
df_count = df_pm.groupby('periodsActivity').size()
df_count = pd.DataFrame(df_count.sort_index())
df_count.columns=['number']
print(df_count.iloc[:50])

## Cluster


### K-Modes clustering method

K-modes clustering is an unsupervised machine learning algorithm designed specifically for categorical data. It is an extension of the [K-means clustering algorithm](https://en.wikipedia.org/wiki/K-means_clustering), which works only with numerical data.

While K-means calculates the mean of data points to find cluster centers (centroids) and uses Euclidean distance (i.e.  linear distances in the geometric space), K-modes adapts this process for categories:
* It uses the mode of the data points in a cluster (the most frequent category) to define the cluster center, rather than the arithmetic mean.
* Simple Matching Distance: It measures dissimilarity between data points by counting the number of mismatched categories (e.g., if two people have different colors, the distance increases by 1), rather than calculating geometric distance.
* Frequency-based Updates: The algorithm iteratively updates cluster modes based on the frequency of categories within each cluster.

[GitHub repository documentation of K-Modes](https://github.com/nicodv/kmodes)


### Choice of the variables

We have to decide which qualitative variables we will use for clustering and experiment with different situations.

&nbsp;

The approach we will test here is to use four basic features of the population: gender, country of origin, secondary (a more precise occupation then physicist/astronomers) and employer class, in order to create clusters.

We exclude the 'generation of activity' variable when clustering, but use it to situate the cluster within the flow of time, and to check whether some profiles are specific to a particular moment in time.

Also, in a first test, we also used the main occupation variable for clustering but this appeared to be too dominant and was hiding more interesting phenomena: cf. this [representation of the clusters](images/kmodes_clusters_with_main_occupation.png) using the main occupation (astronomy/physics). 

We therefore exclude it from clustering and use it to inspect if some clusters are more specific to a main discipline



In [ ]:
### Data to be clustered

categorical_columns=['gender', 'coded_country', 'occupation_sec1', 'coded_employer']
data_cat = df_pm[categorical_columns].values
print(data_cat[:5])

### Determine Optimal Clusters (Elbow Method based on Cost)

N.B.: ou can SKIP this part and directly use 16 and 32 clusters or other multiples of the number of your periods.

&nbsp;

In this part we explore different numbers of clusters in order to discover those that express better the diversity of the population, and the relation to time.

In K-Modes, the cost is the sum of mismatches between every individual and their assigned cluster center (mode).

According to the Elbow Method, we add more clusters as long as the cost lowers significantly. 

You can experiment with different numbers of clusters, but this will take some time.

In [ ]:
### 3. Determine Optimal Clusters (Elbow Method based on Cost)
# can take 20 minutes or more

### you can SKIP this step and directly test 16 and 32 clusters or other multiples of your periods

costs = []
k_range = range(40, 73) # Your specified range: (16, 41) ; (40, 57); (40,73)
for k in k_range:
    km = KModes(n_clusters=k, init='Huang', n_init=20, verbose=0)
    km.fit_predict(data_cat)
    costs.append(km.cost_)

In [ ]:
# Display using a file path
fig_address="doc_images/kmodes_elbow_16-40.jpg"
display(Image(filename=fig_address))

# Or display with specific dimensions
#display(Image(filename=fig_address, width=400, height=300))

In [ ]:
# Plot Elbow
plt.plot(k_range, costs, 'bx-')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Cost (Disagreement)')
plt.title('Elbow Method for Optimal k')
fig_address="doc_images/kmodes_elbow_40-72.jpg"
plt.savefig(fig_address, dpi=100, bbox_inches='tight')
plt.show()

## Define starting clusters manually

In [ ]:
df_vact_g=pd.DataFrame(df_pm.groupby(by=['gender', 'coded_country', 'occupation_main', 'occupation_sec1', 'coded_employer']).size())
df_vact_g.columns=['number']
df_vact_g=df_vact_g.reset_index()
df_vact_g.sort_values(by='number', ascending=False).iloc[:64]

In [ ]:
### group by and count according to sets containing females
df_vact_g=pd.DataFrame(df_pm.groupby(by=['gender', 'coded_country', 'occup', 'occup_s', 'empl']).size())
df_vact_g.columns=['number']
df_vact_g=df_vact_g.reset_index()
#df_vact_g[df_vact_g.gender=='female'].sort_values(by='number', ascending=False).iloc[:64]
df_vact_g_f=df_vact_g[(df_vact_g.gender=='female') & (df_vact_g. number > 2)].sort_values(by='number', ascending=False)
print(len(df_vact_g_f))
df_vact_g_f

### Centroids and cost in K-Modes


The machine learning process calculates the distance of each individual in a cluster from the one individual that was chosen as the centroid or mode in the cluster.

The cost (often called the Total Dissimilarity or Objective Function) is the sum of mismatches between every individual and their assigned cluster center (mode).

In [ ]:
# Select k based on elbow (e.g., let's assume 30 for this run)
optimal_k = 44
# 8, cost: 9043.0;   
# 15, cost: 7890.0;  
# 16, cost: 7719.0;
# 21, cost : 7268.0.0 or 7252 (ML, not only the same )
# 32, cost: 6434.0
# 40, cost: 5953.0
# 44, cost: 5635.0
# 48, cost : 5704.0
# 54, cost : 5342.0
# 64, cost: 4921.0


# 4. Fit Final Model
km_final = KModes(n_clusters=optimal_k, init='Huang', n_init=20, verbose=0)

### We insert here the id of the cluster into a new column of the original dataframe
# The index order is the same, it works
df_pm['cluster'] = km_final.fit_predict(data_cat)
print(km_final.cost_)

In [ ]:
### Inspect the cluster id number at the end of the rows
df_pm.head(3)


### Centroids (modes) and frequencies

Centroids in each cluster are the mode that is at the center of the population. Individuals in the same cluster but with different categories have some distance from this point.

We explore here the features of the 'central' individuals and what the other are.

In [ ]:
### Get the centroids of the clusters
columns = ['gender', 'coded_country', 'occupation_sec1', 'coded_employer']
centroids_array = km_final.cluster_centroids_
centroid_df = pd.DataFrame(centroids_array, 
                           columns=columns)
centroid_df['cluster']=centroid_df.index
print(len(centroid_df))
centroid_df.head()

In [ ]:
### size of the clusters
dfg = pd.DataFrame(df_pm.groupby(by=['cluster']).size())
dfg.columns=['number_in_cl']
dfg.sort_values(by='number_in_cl', ascending=False).head()

In [ ]:
### 
c = dfg.sort_values(by='number_in_cl', ascending=False).plot(kind='bar')
c.tick_params(axis='x', labelsize=7, rotation=70)

In [ ]:
print(dfg.number_in_cl.describe())

#### Inspect frequency of individuals identical with centroid

In [ ]:
### this dataframe contains only rows that are identical with the centroid in each cluster
merged_df = df_pm.merge(centroid_df, on=['gender', 'coded_country', 'occupation_sec1', 'coded_employer', 'cluster'], how='inner')

In [ ]:
### Persons with centroid value in proportion to population

# only persons identical to centroids of their cluster
print(len(merged_df))

# all the persons
print(len(df_pm))

# frequency
print(str(round((len(merged_df)/len(df_pm)*100),1))+' %')


In [ ]:
### Count per cluster how many rows same as centroid
dfcen = pd.DataFrame(merged_df.groupby(by=['cluster']).size())
dfcen.columns=['n_centroid']
dfcen.sort_values(by='n_centroid', ascending=False).head()

#### Inspection of features

In [ ]:
dfgf = pd.DataFrame(df_pm[df_pm.gender=='female'].groupby(by=['cluster']).size())
dfgf.columns=['number_f']
(print(dfgf.sort_values(by='number_f', ascending=False).iloc[:30]))

In [ ]:
### Group and count countries in clusters

# countries per cluster
result_df = pd.DataFrame(df_pm.groupby(by=['cluster', 'coded_country']).size()).reset_index()
result_df.columns=['cluster','coded_country','number']
result_df=result_df.sort_values(['cluster','number'], ascending=[True,False])
result_df.head()


# aggregated countries per cluster
dfg_country = result_df.groupby('cluster')[['coded_country', 'number']].apply(
    lambda x: x.values.tolist()
).reset_index(name='aggregated_data')

### Take the first five per cluster
dfg_country['countries_list'] = dfg_country.aggregated_data.apply(
    lambda lst: ', '.join([f"{item[0]}: {item[1]}" for item in lst[:5]])
)
dfg_country=dfg_country.drop(columns=['cluster','aggregated_data'])


with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', None):
    display(dfg_country.head())

In [ ]:
### Group and count occupations in clusters

# countries per cluster
result_df = pd.DataFrame(df_pm.groupby(by=['cluster', 'occupation_sec1']).size()).reset_index()
result_df.columns=['cluster','occupation_sec1','number']
result_df=result_df.sort_values(['cluster','number'], ascending=[True,False])


# aggregated occupations per cluster
dfg_occupation = result_df.groupby('cluster')[['occupation_sec1', 'number']].apply(
    lambda x: x.values.tolist()
).reset_index(name='aggregated_data')

### Take the first five per cluster
dfg_occupation['occ_list'] = dfg_occupation['aggregated_data'].apply(
    lambda lst: ', '.join([f"{item[0]}: {item[1]}" for item in lst[:5]])
)
dfg_occupation=dfg_occupation.drop(columns=['cluster','aggregated_data'])

with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', None):
    display(dfg_occupation.head())


In [ ]:
### Group and count employer classes in clusters

# countries per cluster
result_df = pd.DataFrame(df_pm.groupby(by=['cluster', 'coded_employer']).size()).reset_index()
result_df.columns=['cluster','coded_employer','number']
result_df=result_df.sort_values(['cluster','number'], ascending=[True,False])


# aggregated occupations per cluster
dfg_empl = result_df.groupby('cluster')[['coded_employer', 'number']].apply(
    lambda x: x.values.tolist()
).reset_index(name='aggregated_data')

### Take the first five per cluster
dfg_empl['empl_list'] = dfg_empl['aggregated_data'].apply(
    lambda lst: ', '.join([f"{item[0]}: {item[1]}" for item in lst[:5]])
)
dfg_empl=dfg_empl.drop(columns=['cluster','aggregated_data'])

with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', None):
    display(dfg_empl.head())


#### Join and inspect

Aggregating the former tables and counts allows to inspect the clusters and identifying the structuring elements of the clusters 

In [ ]:
### Join the different dataframes using the default joining on indexes
# If an error appears in joining, restart the whole process from cenroid_df onward

centroid_df=centroid_df.join(dfg)
centroid_df=centroid_df.join(dfcen)
centroid_df=centroid_df.join(dfgf).fillna(0)
centroid_df['number_f'] = centroid_df['number_f'].astype(int)
centroid_df['prop_f'] = centroid_df.apply(lambda x: (x['number_f']/x['number_in_cl']), axis=1)
centroid_df['prop_f']= centroid_df['prop_f'].round(2)

centroid_df=centroid_df.join(dfg_country)
centroid_df=centroid_df.join(dfg_occupation)
centroid_df=centroid_df.join(dfg_empl)

In [ ]:
a = """
# Align all columns to the left
df.style.set_properties(**{'text-align': 'left'})

# Align specific columns to the left
df.style.set_properties(subset=['column_name_1', 'column_name_2'], **{'text-align': 'left'})

# Align only string columns to the left, keep numbers right (common practice)
df.style.set_properties(**{'text-align': 'left'}, subset=df.select_dtypes(include=['object', 'string']).columns)
"""

In [ ]:
centroid_df['label'] = centroid_df.apply(
    lambda x: str(x.name) + "_" + "_".join(x[['gender', 'coded_country', 'occupation_sec1', 'coded_employer']].astype(str)), 
    axis=1
)
centroid_df.head(1)


In [ ]:
centroid_df['prop_centr']=centroid_df.apply(lambda x : x['n_centroid']/x['number_in_cl'], axis=1).round(2)

In [ ]:
centroid_df=centroid_df[['gender', 'coded_country', 'occupation_sec1', 'coded_employer',
                            'cluster', 'number_in_cl', 
                              'n_centroid', 'prop_centr',
                              'number_f', 'prop_f', 'label',
                              'countries_list', 'occ_list', 
                               'empl_list']]

In [ ]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', None):
    display(centroid_df.sort_values(by='prop_centr', ascending=False).style.set_properties(subset=['occ_list', 'countries_list'], **{'text-align': 'left'}))

In [ ]:
### write centroids to database table for more deep inspection

# N.B this is optional !

db = '../../data/data_analysis.db'

#table_name='clusters_kmodes_centroids_c54'
table_name='kmodes_clusters_centroids'

#centroid_df['run']='cen16'
#centroid_df['run']='cen32'
centroid_df['run']='cen44'
#centroid_df['run']='cen64'


conn = sql.connect(db)
# commented for safety, uncomment to execute
centroid_df.to_sql(table_name, conn, if_exists='append', index=True)
conn.close()

In [ ]:
pers_cluster=df_pm[['person_uri', 'cluster']]
pers_cluster.head()

In [ ]:
### write clusters to database table for more deep inspection

# N.B this is optional !


db = '../../data/data_analysis.db'

table_name='kmodes_clusters'

pers_cluster['run']='cen16'
# pers_cluster['run']='cen32'
# pers_cluster['run']='cen64'

conn = sql.connect(db)
# cursor = conn.cursor()
# pers_cluster.to_sql(table_name, conn, if_exists='append', index=True)
conn.close()

### Is there a correspondence between clusters and generations

In [ ]:
# 5. Validate against Generation (Unused Variable)
# Cross-tabulation : contingency_table
observed = pd.crosstab(df_pm['cluster'], df_pm['periodsActivity'])


In [ ]:
bl.check_chi_square_test_validity(observed)

In [ ]:
expected=bl.bivariate_stats(observed)

### CA

In [ ]:
afc = fa.CA(row_labels=observed.index,col_labels=observed.columns)
afc.fit(observed.values)

In [ ]:
### Inertia (Phi-square - Eigenvalue):  0.108
cal.print_eigenvalue(afc)

In [ ]:
cal.dim_contributions(afc)

In [ ]:
# Represent dimension 1 and 2
afc.mapping(num_x_axis=1,num_y_axis=2,figsize=(8,8))

In [ ]:
# Represent dimension 3 and 4
# afc.mapping(num_x_axis=3,num_y_axis=4,figsize=(8,8))

### Inspection of the clusters

* Use the SQLite tables, cf. this [SQL queries document](../../documentation/wikidata/data-analysis/da5-explore-clusters.sql)
* Use the representation in form of a graph (cf. below)

In [ ]:
width= optimal_k
pp = bl.plot_chi2_residuals(observed.T, figsize=(optimal_k, 7))

In [ ]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', None):
    display(centroid_df[['label','number_in_cl','n_centroid','number_f']])

### Représenter les clusters sous forme de graphes ou d'images  


On exporte les clusters sous forme de graphe au format Gephi: gexf et on les importe dans Gephi LIte Online, où on peut améliorer la représentation et inspecter.




In [ ]:
df_pm.head(2)

In [ ]:
## On peut aussi créer des images, mais très larges si plusieur clusters
columns = ['gender', 'coded_country', 'occupation_sec1', 'coded_employer']
pict_address='images/kmodes_clusters_4_variables_44cl.png'
cf.plot_cluster_networks(df_pm, columns, optimal_k, 
        pict_address=pict_address)

## Do not use following

In [ ]:
## On export 
# output_filename='cluster_graphs/kmodes_clusters_4_variables_32cl.gexf'
output_filename='cluster_graphs/kmodes_clusters_4_variables_44cl.gexf'
categorical_columns = ['gender', 'coded_country', 'occupation_sec1', 'coded_employer']

exp=cf.export_sep_cluster_networks_to_gephi_new(df_pm, categorical_columns, output_filename)